___
# <center>Probabilidade e incerteza</center>
___

## Aula 07

**Objetivo da aula:** ao final desta aula, você deve ser capaz de:

 * calcular a probabilidade de um evento como proporção na base;
 * montar a tabela de dupla entrada e separar conjunta de marginal;
 * calcular uma probabilidade condicional filtrando e recontando;
 * verificar independência comparando o observado com o esperado;
 * aplicar o teorema de Bayes para inverter uma condicional.

Este notebook é curto de propósito. Tudo que está aqui já foi feito na lousa
hoje.

Uma diferença em relação ao slide: lá a tabela de 400 sentenças era
**ilustrativa**, com números escolhidos para fechar de cabeça. Aqui a base é
**real**, e por isso as divisões não dão números redondos. É assim que a conta
aparece na vida.


___
<div id="indice"></div>

## Índice

- [A base de hoje](#dados)

- [Probabilidade é uma proporção](#proporcao)

- [A tabela de dupla entrada](#tabela)

- [Probabilidade condicional](#condicional)

- [Independência: o teste](#independencia)

- [Teorema de Bayes](#bayes)

- [RESUMO](#resumo)


___
<div id="dados"></div>

# A base de hoje

Acórdãos do TJSP em ações contra planos de saúde. Uma linha por acórdão.


In [ ]:
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

URL = "https://raw.githubusercontent.com/jtrecenti/202662-cdad2/main/dados"

saude = pd.read_csv(f"{URL}/tjsp_cjsg_plano_saude.csv")

saude.shape


Duas colunas de sim ou não interessam hoje:

- `tem_dano_moral`: o acórdão reconheceu dano moral;
- `houve_majoracao`: o valor da indenização foi majorado.


In [ ]:
saude[["tem_dano_moral", "houve_majoracao"]].head()


[Volta ao Índice](#indice)


___
<div id="proporcao"></div>

# Probabilidade é uma proporção

$P(A)$ é o número de casos em $A$ dividido pelo total. No pandas, uma coluna de
`True` e `False` já sabe fazer essa conta: a média de uma coluna booleana **é**
a proporção de `True`.


In [ ]:
dano = saude["tem_dano_moral"]

dano.mean()


Por que a média funciona: `True` vale 1 e `False` vale 0, então somar a coluna
conta os casos e dividir pelo tamanho dá a proporção.


**✍️ Agora você.** Calcule $P(\text{houve majoração})$. Mesma ideia, outra coluna.


In [ ]:
majorou = saude["houve_majoracao"]

majorou.mean()


[Volta ao Índice](#indice)


___
<div id="tabela"></div>

# A tabela de dupla entrada

`pd.crosstab` cruza duas colunas e conta quantos casos caem em cada combinação.

Os dois nomes da aula aparecem aqui:

- o **miolo** é a distribuição **conjunta**, e cada célula responde por duas
  variáveis ao mesmo tempo;
- a última linha e a última coluna são as **marginais**, e cada valor delas
  responde por uma variável só.


✔️ **Uso do `pd.crosstab`**

```python
# Sintaxe geral:
pd.crosstab(coluna_das_linhas, coluna_das_colunas, margins=True)
```

Documentação oficial: [pd.crosstab](https://pandas.pydata.org/docs/reference/api/pandas.crosstab.html)


In [ ]:
pd.crosstab(
    saude["tem_dano_moral"],
    saude["houve_majoracao"],
    margins=True,
    margins_name="total",
)


✔️ **A mesma tabela em proporções.** `normalize="all"` divide tudo pelo total
geral, e é a versão do slide em que o canto vale 1.


In [ ]:
pd.crosstab(
    saude["tem_dano_moral"],
    saude["houve_majoracao"],
    margins=True,
    margins_name="total",
    normalize="all",
).round(3)


[Volta ao Índice](#indice)


___
<div id="condicional"></div>

# Probabilidade condicional

Condicionar é trocar o denominador: em vez de dividir pelo total, dividimos só
pelo grupo que interessa. No pandas isso é **filtrar e recontar**.


In [ ]:
# Só os acórdãos que reconheceram dano moral
com_dano = saude.query("tem_dano_moral")

len(com_dano)


In [ ]:
# P(majoração | tem dano moral)
com_dano["houve_majoracao"].mean()


Compare com a proporção de majoração na base inteira. Saber que houve dano
moral **muda** o número, e é isso que significa dizer que as duas variáveis têm
relação.


**✍️ Agora você.** Agora o outro lado: $P(\text{majoração} \mid \text{sem dano moral})$.


In [ ]:
sem_dano = saude.query("~tem_dano_moral")

sem_dano["houve_majoracao"].mean()


⚠️ **Cuidado com o `normalize`.** `"index"` divide por linha, `"columns"` por
coluna e `"all"` pelo total. As três dão números diferentes e respondem a
perguntas diferentes. Errar aqui é o mesmo erro de dividir pelo total em vez de
pelo grupo.


In [ ]:
pd.crosstab(
    saude["tem_dano_moral"],
    saude["houve_majoracao"],
    normalize="index",
).round(3)


[Volta ao Índice](#indice)


___
<div id="independencia"></div>

# Independência: o teste

Se dois eventos fossem independentes, a probabilidade dos dois juntos seria o
produto das marginais. O teste é comparar esse produto com o que a base tem.


In [ ]:
esperado = dano.mean() * majorou.mean() * len(saude)
observado = (dano & majorou).sum()

print("esperado sob independência:", round(esperado, 1))
print("observado                 :", observado)


Cerca de 15 contra 21. A diferença é o tamanho da associação entre reconhecer
dano moral e majorar o valor.

Na aula 17 vamos aprender a decidir se uma diferença dessas é grande o bastante
para não ser acaso. Aqui, com números pequenos, a cautela é ainda maior.


[Volta ao Índice](#indice)


___
<div id="bayes"></div>

# Teorema de Bayes

Bayes inverte a condicional:

$$P(A \mid B) = \frac{P(B \mid A)\,P(A)}{P(B)}$$

Vamos conferir que ele bate com a conta direta.


In [ ]:
p_dano = dano.mean()
p_maj = majorou.mean()
p_maj_dado_dano = com_dano["houve_majoracao"].mean()

# Bayes: P(dano | majorou)
bayes = p_maj_dado_dano * p_dano / p_maj

# a conta direta, filtrando
direto = saude.query("houve_majoracao")["tem_dano_moral"].mean()

print("por Bayes :", round(bayes, 4))
print("direto    :", round(direto, 4))


Os dois dão o mesmo número, e é assim que tem que ser: Bayes não é uma conta
nova, é a regra do produto escrita de outro jeito.

Ele importa quando você **não tem a base inteira** para filtrar, e só conhece
$P(B \mid A)$ e as marginais. É a situação da perícia: o laudo informa a
taxa de erro do exame, e ninguém tem a tabela do lote inteiro.


<div id="ex1"></div>

### EXERCÍCIO 1

A perícia grafotécnica da aula, agora em código.

Dois eventos, e só eles:

- $F$: a assinatura do contrato **é falsa**;
- $A$: a perícia **aponta** falsidade nesse contrato.

O enunciado da lousa dá três números:

- $P(F) = 0{,}01$, porque 1 em cada 100 contratos do lote tem assinatura falsa;
- $P(A \mid F) = 0{,}99$, porque **entre os contratos falsos** a perícia aponta
  em 99% das vezes;
- $P(A \mid F^c) = 0{,}01$, porque **entre os autênticos** ela aponta em 1% das
  vezes, por variação natural da assinatura.

E pede $P(F \mid A)$: a perícia apontou este contrato, qual a probabilidade de
a assinatura ser mesmo falsa. É a inversão.

1. escreva os três números como variáveis;
2. calcule $P(A)$ pela lei da probabilidade total;
3. calcule $P(F \mid A)$ por Bayes, e confira com os 50% da lousa;
4. refaça com $P(F) = 0{,}50$, como se a perícia só fosse pedida em contratos já
   sob suspeita. O que acontece com a resposta?


In [ ]:
def p_falso_dado_apontado(p_falso,
                          p_aponta_dado_falso=0.99,
                          p_aponta_dado_autentico=0.01):
    # lei da probabilidade total: os apontados saem dos falsos e dos autênticos
    p_aponta = (p_aponta_dado_falso * p_falso
                + p_aponta_dado_autentico * (1 - p_falso))
    # Bayes
    return p_aponta_dado_falso * p_falso / p_aponta

for antes in (0.01, 0.10, 0.50):
    print(f"P(F) = {antes:.0%}  ->  P(F | A) = {p_falso_dado_apontado(antes):.1%}")


💡 A perícia é a mesma nas três linhas, e o laudo diria exatamente a mesma
coisa. O que muda é **em que lote ela foi aplicada**. Por isso periciar o lote
inteiro e periciar só os contratos já sob suspeita são decisões diferentes, com
o mesmo perito e o mesmo equipamento.


[Volta ao Índice](#indice)


___
<div id="resumo"></div>

# RESUMO

| ideia | no pandas |
|---|---|
| $P(A)$ | média de uma coluna booleana |
| tabela de dupla entrada | `pd.crosstab(a, b, margins=True)` |
| a mesma tabela em proporções | `normalize="all"` |
| $P(A \mid B)$ | `.query()` no B, e a média de A dentro do filtro |
| todas as condicionais de uma vez | `normalize="index"` ou `"columns"` |
| independência | comparar $P(A)P(B)n$ com o observado |
| Bayes | $P(B \mid A)P(A)/P(B)$, com $P(B)$ pela marginal |

**A frase para levar:** condicionar é trocar o denominador, e Bayes é o que
permite trocar de volta.


[Volta ao Índice](#indice)
